### Sistema di log per infrastrutture IT

In [ ]:
from influxdb_client import InfluxDBClient, BucketsApi, BucketRetentionRules, Point, WritePrecision
from datetime import datetime, timedelta, timezone
import psutil
import numpy as np
import random, time
import pandas as pd
from sklearn.cluster import KMeans
from river import cluster
import matplotlib.pyplot as plt

# Setup e connessione al server InfluxDB
org_name = "spamic"
token = "6ANpdrZyP8Nk7HBib5puP7dx8fJrFNG2xCPlBiXqDSKAsSfjKsUMPlgEVvoQMllyP0HCDAlherDBZZA_z42utw=="
url = "http://localhost:8086"

In [ ]:
# Creazione bucket da codice
def create_bucker(client, bucket_name, retention_seconds = None):
    """
    Crea un nuovo bucket in InfluxDB con una retention rule opzionale.
    
    :param client: Istanza di InfluxDBClient già autenticata.
    :param bucket_name: Nome del bucket da creare
    :param retention_seconds: Durata della retention rule in secondi (se None = nessuna retention)
    :return: Oggetto Bucket creato o None in caso di errore.
    """
    try:
        buckets_api = client.buckets_api()
        bucket = buckets_api.find_bucket_by_name(bucket_name)

        if bucket:
            print(f"Bucket {bucket_name} already exists")
        else:
            if retention_seconds:
                retention_rules = [BucketRetentionRules(type="expire",every_seconds=retention_seconds)]
            else:
                retention_rules = []
            
            bucket = buckets_api.create_bucket(
                bucket_name=bucket_name,
                retention_rules=retention_rules,
                org=org_name
            )

            print(f"Bucket {bucket_name} created")

    except Exception as e:
        print(f"Error during {bucket_name} creation: {e}")

In [ ]:
bucket_name = "system_log"
client = InfluxDBClient(url=url, token=token, org=org_name)
write_api = client.write_api()

In [ ]:
create_bucker(client, bucket_name, retention_seconds=30*24*60*60)

In [ ]:
def get_metrics():
    """ Get system metrics by psutil """
    cpu_percent = psutil.cpu_percent(interval=1)
    memory = psutil.virtual_memory().percent
    uptime = time.time() - psutil.boot_time()

    return cpu_percent, memory, uptime

def write_metrics(cpu_percent, memory, uptime):
    """ Write metrics """
    point = (
        Point("system_metrics")
        .field("cpu_percent", cpu_percent)
        .field("memory", memory)
        .field("uptime", uptime)
        .tag("host", "Asus N580GD")
        .time(time.time_ns(), WritePrecision.NS)
    )
    write_api.write(bucket=bucket_name, org=org_name, record=point)

client.close()

In [ ]:
start_time = time.time()
duration = 30

def timedelta_to_iso(td):
    total_seconds = int(td.total_seconds())
    hours, remainder = divmod(total_seconds, 3600)
    minutes, seconds = divmod(remainder, 60)
    return f"{hours} ore {minutes} minuti {seconds} secondi"

try:
    while time.time() - start_time < duration:
        cpu_percent, memory, uptime = get_metrics()

        uptime_td = timedelta(seconds=int(uptime))
        uptime_iso = timedelta_to_iso(uptime_td)

        print(f"CPU: {cpu_percent}%, Memory: {memory}%, Uptime: {uptime_iso}")

        write_metrics(cpu_percent, memory, uptime)
        time.sleep(0.01)

except KeyboardInterrupt:
    print("Interruzione da tastiera. Uscita...")

In [ ]:
client = InfluxDBClient(url=url, token=token, org=org_name)
query  = f'''
    from(bucket: "{bucket_name}") 
    |> range(start: -10d) 
    |> filter(fn: (r) => r._measurement == "system_metrics")
    '''
tables = client.query_api().query(query,org=org_name)
for table in tables:
    for record in table.records:
        field = record.get_field()
        value = record.get_value()

        #Se il campo è uptime, converto da secondi a ore, minuti e secondi
        if field == "uptime":
            value = timedelta_to_iso(timedelta(seconds=value))
        
        print(f"{field}: {value}")
    
client.close()